In [ ]:
import os
import json
import warnings
import numpy as np
import xarray as xr
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':8,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal'})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
PREDSDIR   = CONFIGS['filepaths']['predictions']
SPLIT      = 'test'
TARGETVAR  = 'tp'

with open(os.path.join(SPLITSDIR,'stats.json'),'r',encoding='utf-8') as f:
    STATS = json.load(f)
MEAN = STATS[f'{TARGETVAR}_mean']
STD  = STATS[f'{TARGETVAR}_std']
ZMIN = (0.0 - MEAN) / STD

In [ ]:
def to_native(zscore):
    return np.maximum(np.expm1(zscore * STD + MEAN),0.0)

def to_zscore(native_mm):
    return (np.log1p(native_mm) - MEAN) / STD

def r2(ytrue,ypred):
    ss_res = np.nansum((ytrue - ypred)**2)
    ss_tot = np.nansum((ytrue - np.nanmean(ytrue))**2)
    return 1 - ss_res / ss_tot

def loss_plain_mse(raw,y):
    pred = ZMIN + np.maximum(raw,0.0)
    return np.nanmean((pred - y)**2)

def loss_weighted_exp(raw,y):
    pred = ZMIN + np.maximum(raw,0.0)
    return np.nanmean(np.exp(y) * (pred - y)**2)

def loss_weighted_exp2(raw,y):
    pred = ZMIN + np.maximum(raw,0.0)
    return np.nanmean(np.exp(2*y) * (pred - y)**2)

def loss_native_mse(raw,y):
    pred_z = ZMIN + np.maximum(raw,0.0)
    pred_mm = to_native(pred_z)
    true_mm = to_native(y)
    return np.nanmean((pred_mm - true_mm)**2)

def loss_native_mae(raw,y):
    pred_z = ZMIN + np.maximum(raw,0.0)
    pred_mm = to_native(pred_z)
    true_mm = to_native(y)
    return np.nanmean(np.abs(pred_mm - true_mm))

def native_r2(raw,y):
    pred_z  = ZMIN + np.maximum(raw,0.0)
    pred_mm = to_native(pred_z)
    true_mm = to_native(y)
    return r2(true_mm,pred_mm)

LOSSFNS = {
    'Z-scored MSE\n(current)':loss_plain_mse,
    'Weighted MSE\nexp(y)':loss_weighted_exp,
    'Weighted MSE\nexp(2y)':loss_weighted_exp2,
    'Native mm\nMSE':loss_native_mse,
    'Native mm\nMAE':loss_native_mae}

In [ ]:
with xr.open_dataset(os.path.join(SPLITSDIR,f'norm_{SPLIT}.h5'),engine='h5netcdf') as ds:
    ytrue_z = ds[TARGETVAR].transpose('time','lat','lon').values
with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    ytrue_mm = ds[TARGETVAR].transpose('time','lat','lon').values

validmask = np.isfinite(ytrue_z.ravel()) & np.isfinite(ytrue_mm.ravel())
yz = ytrue_z.ravel()[validmask]
ymm = ytrue_mm.ravel()[validmask]
print(f'Loaded {SPLIT} split: {validmask.sum():,} valid samples, zmin={ZMIN:.4f}')

In [ ]:
def evaluate_pareto(predpath,yz,ymm,validmask,shape):
    with xr.open_dataset(predpath) as ds:
        predtp = ds[TARGETVAR].load()
    if 'seed' in predtp.dims:
        predtp = predtp.mean('seed')
    records = []
    for c in predtp.complexity.values:
        pred_mm = predtp.sel(complexity=int(c)).values.ravel()[validmask]
        raw = to_zscore(pred_mm) - ZMIN
        row = {'complexity':int(c),'native_r2':r2(ymm,pred_mm)}
        for lname,lfn in LOSSFNS.items():
            row[lname] = lfn(raw,yz)
        records.append(row)
    return records

def evaluate_seeds(predpath,yz,ymm,validmask):
    with xr.open_dataset(predpath) as ds:
        predtp = ds[TARGETVAR].load()
    records = []
    for s in predtp.seed.values:
        pred_mm = predtp.sel(seed=int(s)).values
        if 'complexity' in predtp.dims:
            pred_mm = pred_mm[...,0]
        pred_mm = pred_mm.ravel()[validmask]
        raw = to_zscore(pred_mm) - ZMIN
        row = {'seed':int(s),'native_r2':r2(ymm,pred_mm)}
        for lname,lfn in LOSSFNS.items():
            row[lname] = lfn(raw,yz)
        records.append(row)
    return records

In [ ]:
srbl_path  = os.path.join(PREDSDIR,f'sr_bl_{SPLIT}_predictions.nc')
srbl_records = evaluate_pareto(srbl_path,yz,ymm,validmask,ytrue_z.shape)
print(f'SR-BL: {len(srbl_records)} complexity levels')
for rec in srbl_records:
    print(f"  C={rec['complexity']:2d}  R2={rec['native_r2']:.4f}  z-MSE={rec['Z-scored MSE\n(current)']:.4f}")

In [ ]:
nngauss_path = os.path.join(PREDSDIR,f'nn_gauss_{SPLIT}_predictions.nc')
nngauss_records = evaluate_seeds(nngauss_path,yz,ymm,validmask)
print(f'NN-GAUSS: {len(nngauss_records)} seeds')
for rec in nngauss_records:
    print(f"  seed={rec['seed']}  R2={rec['native_r2']:.4f}  z-MSE={rec['Z-scored MSE\n(current)']:.4f}")

In [ ]:
sratm_path = os.path.join(PREDSDIR,f'sr_atm_{SPLIT}_predictions.nc')
srsfc_path = os.path.join(PREDSDIR,f'sr_sfc_{SPLIT}_predictions.nc')
srall_path = os.path.join(PREDSDIR,f'sr_all_{SPLIT}_predictions.nc')

extra_pareto = {}
for label,path in [('SR-ATM',sratm_path),('SR-SFC',srsfc_path),('SR-ALL',srall_path)]:
    if os.path.exists(path):
        extra_pareto[label] = evaluate_pareto(path,yz,ymm,validmask,ytrue_z.shape)
        print(f'{label}: {len(extra_pareto[label])} complexity levels')
    else:
        print(f'{label}: predictions not found')

In [ ]:
def rank_correlation(records,lossname):
    from scipy.stats import spearmanr
    losses = [r[lossname] for r in records]
    r2s    = [r['native_r2'] for r in records]
    if len(set(losses)) < 3:
        return np.nan
    corr,_ = spearmanr(losses,r2s)
    return corr

print('Spearman rank correlation between each loss and native R^2')
print('(should be -1.0: lower loss = higher R^2)\n')
datasets = {'SR-BL':srbl_records}
datasets.update(extra_pareto)
for dname,records in datasets.items():
    if len(records) < 3:
        continue
    print(f'{dname} ({len(records)} equations):')
    for lname in LOSSFNS:
        rho = rank_correlation(records,lname)
        print(f'  {lname.replace(chr(10)," "):25s}  rho = {rho:+.3f}')
    print()

In [ ]:
def plot_loss_vs_r2(records,title,idlabel='complexity'):
    lossnames = list(LOSSFNS.keys())
    ncols = len(lossnames)
    fig,axs = pplt.subplots(ncols=ncols,figwidth=10,share=False,tight=True)
    for ax,lname in zip(axs,lossnames):
        losses = [r[lname] for r in records]
        r2s    = [r['native_r2'] for r in records]
        ids    = [r[idlabel] for r in records]
        ax.scatter(losses,r2s,color='#2d7d9a',s=30,zorder=5)
        for i,cid in enumerate(ids):
            ax.text(losses[i],r2s[i]+0.005,str(cid),ha='center',va='bottom',fontsize=6)
        rho = rank_correlation(records,lname)
        ax.text(0.05,0.95,f'rho={rho:+.2f}',transform=ax.transAxes,va='top',fontsize=8)
        ax.format(xlabel=lname.replace('\n',' '),ylabel='Native mm R²' if ax==axs[0] else '',grid=False)
    fig.suptitle(title)
    return fig

fig = plot_loss_vs_r2(srbl_records,'SR-BL Pareto: Loss metric vs native R²')
pplt.show()

In [ ]:
for dname,records in extra_pareto.items():
    if len(records) < 3:
        continue
    fig = plot_loss_vs_r2(records,f'{dname} Pareto: Loss metric vs native R²')
    pplt.show()

In [ ]:
if len(nngauss_records) >= 3:
    fig = plot_loss_vs_r2(nngauss_records,'NN-GAUSS seeds: Loss metric vs native R²',idlabel='seed')
    pplt.show()
else:
    print(f'NN-GAUSS: only {len(nngauss_records)} seeds, need >=3 for correlation')
    for rec in nngauss_records:
        print(f"  seed={rec['seed']}  R2={rec['native_r2']:.4f}")
        for lname in LOSSFNS:
            print(f"    {lname.replace(chr(10),' '):25s} = {rec[lname]:.6f}")

In [ ]:
print('Summary: which loss would PySR use to pick the best equation?\n')
for dname,records in datasets.items():
    if len(records) < 3:
        continue
    r2s = [r['native_r2'] for r in records]
    best_r2_idx = np.argmax(r2s)
    print(f'{dname}: best native R²={r2s[best_r2_idx]:.4f} at C={records[best_r2_idx].get("complexity","?")}') 
    for lname in LOSSFNS:
        losses = [r[lname] for r in records]
        best_loss_idx = np.argmin(losses)
        match = 'MATCH' if best_loss_idx == best_r2_idx else f'MISMATCH (picks C={records[best_loss_idx].get("complexity","?")}, R2={r2s[best_loss_idx]:.4f})'
        print(f'  {lname.replace(chr(10)," "):25s} -> {match}')
    print()